#  Semantic Search Engine

## Knowledge base on history of Maharashtra

In [39]:
sentencel=['Ancient Maharashtra was governed by powerful regional dynasties, including the Mauryas, Satavahanas, and Vakatakas',
 'The breathtaking rock-cut monuments at Ajanta and Ellora Caves were carved during this classical ancient period ',
 'Throughout the medieval period, the region fell under the rule of the Delhi Sultanate and later the Bahmani Sultanate ',
 'In the 17th century, Chhatrapati Shivaji Maharaj carved out an independent Maratha kingdom by uniting the local people ',
 "Shivaji's guerrilla warfare tactics revolutionized regional combat and challenged the might of the Mughal Empire ",
 "Following Shivaji's rule, the Maratha Empire expanded significantly under the administration of the Peshwas ",
 'The Maratha Empire became the dominant power in the Indian subcontinent until it was defeated by the British in the Anglo-Maratha Wars ',
 'During the 19th and early 20th centuries, Maharashtra was a major epicenter for the Indian independence movement, led by figures like Bal Gangadhar Tilak and Gopal Krishna Gokhale ',
 'After India gained independence, the region was temporarily part of Bombay State before the linguistic formation of modern Maharashtra on May 1, 1960',
 "Today, Maharashtra stands as India's premier industrial and economic powerhouse, with its capital in the bustling metropolis of Mumbai"]

## Embeddings are created using Mistral AI

In [52]:
import os
from mistralai.client import Mistral
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env.txt")
api_key = os.getenv("MISTRAL_API_KEY")
client = Mistral(api_key=api_key)

In [41]:
print("Embedding...")
vecs = [item.embedding for item in client.embeddings.create(
    model="mistral-embed", inputs=sentencel
).data]

Embedding...


## ChromaDB

In [45]:
import chromadb
clientdb=chromadb.Client()
collection=clientdb.create_collection(name="history")
doc_ids = [f"doc_{i}" for i in range(len(sentencel))]
collection.add(
    ids=doc_ids,             # unique ID per document
    embeddings=vecs,   # the 1536-number vectors
    documents=sentencel,     # original text saved as metadata
)

print(f"Documents stored in ChromaDB: {collection.count()}")

Documents stored in ChromaDB: 10


## User query with the top 3 most similar result

In [46]:
user_query="Who founded the Maratha Empire in the 17th century?"
query_embedding=client.embeddings.create(model="mistral-embed" , inputs=user_query).data[0].embedding
results=collection.query(query_embeddings=[query_embedding],
    n_results=3)
results['documents'][0]

['In the 17th century, Chhatrapati Shivaji Maharaj carved out an independent Maratha kingdom by uniting the local people ',
 'The Maratha Empire became the dominant power in the Indian subcontinent until it was defeated by the British in the Anglo-Maratha Wars ',
 "Following Shivaji's rule, the Maratha Empire expanded significantly under the administration of the Peshwas "]

In [47]:
queries = [
    "Shirdi Sai Baba famously made his permanent home inside a dilapidated, local village mosque. What specific name did he give to this mosque to symbolise religious harmony?",
    "When did battle of panipat happened",
    "Which British Plague Commissioner was assassinated in Pune in 1897 by the Chapekar brothers?",
]

query_embedding = client.embeddings.create(
    model="mistral-embed",
    inputs=queries
).data[0].embedding

# Find the 2 most similar documents
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

results['documents'][0]

['After India gained independence, the region was temporarily part of Bombay State before the linguistic formation of modern Maharashtra on May 1, 1960',
 'In the 17th century, Chhatrapati Shivaji Maharaj carved out an independent Maratha kingdom by uniting the local people ',
 'Throughout the medieval period, the region fell under the rule of the Delhi Sultanate and later the Bahmani Sultanate ']

## Cosine Similarity

In [56]:
import numpy as np
def cosine_similarity(senta,sentb):
    v1=np.array(senta)
    v2=np.array(sentb)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
sq=[
    "Chhatrapati Shivaji Maharaj founded the Maratha Empire in the 17th century",
    "The Maratha Empire expanded significantly under the administration of the Peshwas ",
    "Bal Gangadhar Tilak and Gopal Krishna Gokhale were towering 19th-century Indian nationalist leaders from Maharashtra",
    "Mumbai is the capital of India",
     "I love working with cars"
]
embeddings = client.embeddings.create(
    model="mistral-embed",
    inputs=sq
)
emba=embeddings.data[0].embedding
embb=embeddings.data[1].embedding
embc=embeddings.data[2].embedding
embd=embeddings.data[3].embedding
embe=embeddings.data[4].embedding
simaa = cosine_similarity(emba, emba)
simab = cosine_similarity(emba, embb)
simac = cosine_similarity(emba, embc)
simad = cosine_similarity(emba , embd)
simae = cosine_similarity(emba , embe)
print(f"Similarity between {sq[0]} and {sq[0]} is {simaa}")
print(f"Similarity between {sq[0]} and {sq[1]} is {simab}")
print(f"Similarity between {sq[0]} and {sq[2]} is {simac}")
print(f"Similarity between {sq[0]} and {sq[3]} is {simad}")
print(f"Similarity between {sq[0]} and {sq[4]} is {simae}")

Similarity between Chhatrapati Shivaji Maharaj founded the Maratha Empire in the 17th century and Chhatrapati Shivaji Maharaj founded the Maratha Empire in the 17th century is 0.9999999999999998
Similarity between Chhatrapati Shivaji Maharaj founded the Maratha Empire in the 17th century and The Maratha Empire expanded significantly under the administration of the Peshwas  is 0.8575390628308722
Similarity between Chhatrapati Shivaji Maharaj founded the Maratha Empire in the 17th century and Bal Gangadhar Tilak and Gopal Krishna Gokhale were towering 19th-century Indian nationalist leaders from Maharashtra is 0.8136894333730875
Similarity between Chhatrapati Shivaji Maharaj founded the Maratha Empire in the 17th century and Mumbai is the capital of India is 0.7696655772491738
Similarity between Chhatrapati Shivaji Maharaj founded the Maratha Empire in the 17th century and I love working with cars is 0.630192670409146
